In [1]:
import os
os.environ["KERAS_BACKEND"] = "torch"

In [2]:
import torch
from VGG_Arthur import *
from keras.layers import TorchModuleWrapper
from deel import torchlip
import keras
import torchattacks


In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [4]:
model = load_model().to(device)
model.load_state_dict(torch.load('/home/aws_install/robustess_project/lip_notebooks/notebooks_creation_models/Vgg_lip_multisteplr.pt', weights_only=True))
model.eval()

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): ParametrizationList(
        (0): _SpectralNorm()
        (1): _BjorckNorm()
        (2): _LConvNorm()
      )
    )
  )
  (norm): BatchCentering()
  (activation): GroupSort2()
  (scalar): MultiplyByScalar()
)
  warnings.warn(
/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/deel/torchlip/modules/module.py:159: UserWarning: Sequential model contains a layer which is not a Lipschitz layer: LipBlock(
  (conv): ParametrizedSpectralConv2d(
    96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
    (parametrizations): ModuleDict(
      (weight): Pa

Sequential(
  (0): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (1): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (2): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), p

# Keras

In [5]:
class VGG(keras.Model):
    def __init__(self, model):
        super().__init__()
        self.model = model
    def call(self, x):
        return self.model(x)

In [6]:
keras_model = VGG(model=model)

In [7]:
keras_model.summary()

Model: "vgg"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper            │ ?                      │    17,598,730 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,598,730 (67.13 MB)

 Trainable params: 17,598,730 (67.13 MB)

 Non-trainable params: 0 (0.00 B)

In [8]:
(x_train, y_train),_ = keras.datasets.cifar10.load_data()
x_train = x_train.astype("float32") / 255



In [9]:
print(torch.argmax(model(torch.from_numpy(x_train[:10]).to(device)), dim=-1))
print(y_train[:10])
# .to(device)

tensor([6, 9, 9, 4, 1, 1, 2, 7, 8, 3], device='cuda:0')
[[6]
 [9]
 [9]
 [4]
 [1]
 [1]
 [2]
 [7]
 [8]
 [3]]


In [10]:
keras_model.summary()

Model: "vgg"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ torch_module_wrapper            │ ?                      │    17,598,730 │
│ (TorchModuleWrapper)            │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 17,598,730 (67.13 MB)

 Trainable params: 17,598,730 (67.13 MB)

 Non-trainable params: 0 (0.00 B)

In [11]:
# keras_model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
# keras_model.fit(x_train, y_train, epochs=2)

In [12]:
# keras_model.save("VGG_keras.keras")

# Torch

In [13]:
import sys
sys.path.append("..")
from data_processing_torch import *
from torchvision import datasets
from torchvision.transforms import v2
from torch.utils.data import DataLoader
import numpy as np
import yaml

In [14]:
# Load configuration file
with open('config.yaml', 'r') as f:
    cfg = yaml.safe_load(f)

In [15]:
# Load dataset
train_loader, test_loader = load_cifar10(cfg)

/home/aws_install/miniconda3/envs/k3torchenv/lib/python3.10/site-packages/torchvision/transforms/v2/_deprecated.py:42: UserWarning: The transform `ToTensor()` is deprecated and will be removed in a future release. Instead, please use `v2.Compose([v2.ToImage(), v2.ToDtype(torch.float32, scale=True)])`.Output is equivalent up to float precision.
  warnings.warn(


In [16]:
model.eval()
for img, lb in test_loader:
    break
prediction = model(torch.ones_like(img.to(device))*1)
print(prediction)


tensor([[-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        ...,
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017],
        [-0.0008, -0.0007, -0.0013,  ..., -0.0016, -0.0008, -0.0017]],
       device='cuda:0', grad_fn=<MulBackward0>)


In [17]:
images, labels = select_data_for_radius_evaluation(test_loader, test_loader.dataset, model)

In [18]:
model.eval()
model(images[171:172].to(device))

tensor([[-0.0036, -0.0033, -0.0035, -0.0040, -0.0034, -0.0045, -0.0035, -0.0043,
          0.0044, -0.0037]], device='cuda:0', grad_fn=<MulBackward0>)

In [19]:
model.eval()
model(torch.ones_like(images[171:172].to(device))*1)

tensor([[-0.0008, -0.0007, -0.0013, -0.0008, -0.0019, -0.0005, -0.0019, -0.0016,
         -0.0008, -0.0017]], device='cuda:0', grad_fn=<MulBackward0>)

In [20]:
images[171:172]

tensor([[[[0.9529, 0.9373, 0.9294,  ..., 0.9804, 0.9804, 0.9882],
          [0.9451, 0.9216, 0.9216,  ..., 0.9804, 0.9765, 0.9882],
          [0.9529, 0.9255, 0.9137,  ..., 0.9725, 0.9686, 0.9765],
          ...,
          [0.0667, 0.0902, 0.1490,  ..., 0.3098, 0.3529, 0.4314],
          [0.1373, 0.0941, 0.0980,  ..., 0.4118, 0.4431, 0.4510],
          [0.1451, 0.1176, 0.1451,  ..., 0.4314, 0.4824, 0.4588]],

         [[0.9333, 0.9176, 0.9098,  ..., 0.9765, 0.9765, 0.9843],
          [0.9294, 0.9059, 0.9020,  ..., 0.9765, 0.9765, 0.9843],
          [0.9373, 0.9098, 0.8980,  ..., 0.9686, 0.9686, 0.9725],
          ...,
          [0.1333, 0.1529, 0.1961,  ..., 0.3059, 0.3490, 0.4275],
          [0.2078, 0.1725, 0.1569,  ..., 0.4078, 0.4392, 0.4471],
          [0.2039, 0.1882, 0.1961,  ..., 0.4314, 0.4784, 0.4549]],

         [[0.9098, 0.8980, 0.8902,  ..., 0.9608, 0.9608, 0.9686],
          [0.8980, 0.8784, 0.8745,  ..., 0.9529, 0.9490, 0.9608],
          [0.8941, 0.8706, 0.8588,  ..., 0

In [21]:
van_model = model.vanilla_export()
van_model = van_model.to(device)
van_model.eval()

Sequential(
  (0): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      3, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (1): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False, padding_mode=reflect
      (parametrizations): ModuleDict(
        (weight): ParametrizationList(
          (0): _SpectralNorm()
          (1): _BjorckNorm()
          (2): _LConvNorm()
        )
      )
    )
    (norm): BatchCentering()
    (activation): GroupSort2()
    (scalar): MultiplyByScalar()
  )
  (2): LipBlock(
    (conv): ParametrizedSpectralConv2d(
      96, 96, kernel_size=(3, 3), stride=(1, 1), p

In [22]:
# calcul certificat
def compute_certificate(images, model, L=1):    
    values, _ = torch.topk(model(images), k=2)
    certificates = (values[:, 0] - values[:, 1]) / (np.sqrt(2)*L)
    return certificates  

certificates = compute_certificate(images.to(device), model, L=1)

In [23]:
certificates[171]

tensor(0.0054, device='cuda:0', grad_fn=<SelectBackward0>)

In [24]:
model(images[171:172].to(device))

tensor([[-0.0036, -0.0033, -0.0035, -0.0040, -0.0034, -0.0045, -0.0035, -0.0043,
          0.0044, -0.0037]], device='cuda:0', grad_fn=<MulBackward0>)

In [25]:
van_model(images[171:172].to(device))

tensor([[-0.0036, -0.0033, -0.0035, -0.0040, -0.0034, -0.0045, -0.0035, -0.0043,
          0.0044, -0.0037]], device='cuda:0', grad_fn=<MulBackward0>)

In [26]:
# test attaque
eps = 0.65
atk_van = torchattacks.PGDL2(model, eps=eps, alpha=eps/5, steps=int((10*eps)), random_start=True)

In [27]:
adv_image = atk_van(images[171:172], labels[171:172])

In [28]:
torch.argmax(model(adv_image))

tensor(9, device='cuda:0')

In [29]:
torch.linalg.norm(adv_image-images[:1].to(device))

tensor(12.8269, device='cuda:0')

In [30]:
import pickle
# Define the directory and file paths
output_dir = "./../benchmark_dataset"
images_path = os.path.join(output_dir, "images.pkl")
targets_path = os.path.join(output_dir, "targets.pkl")
# --- Load the Tensors ---
print(f"Loading data from {output_dir}...")

# Load the images tensor
with open(images_path, 'rb') as f:
    images = pickle.load(f)

# Load the targets tensor
with open(targets_path, 'rb') as f:
    labels = pickle.load(f)

images = images.to(device)
labels = labels.to(device)

Loading data from ./../benchmark_dataset...


In [31]:
from radius_evaluation_tools_torch import *

In [ ]:
single_compute_optimistic_radius_PGD(171, images.to(device), labels.to(device), certificates, model, n_iter=10)

tensor([0.0054], device='cuda:0', grad_fn=<SliceBackward0>)
> /home/aws_install/robustess_project/lip_notebooks/radius_evaluation_tools_torch.py(58)single_compute_optimistic_radius_PGD()
     56     d_low = 0
     57     pdb.set_trace()
---> 58     print('eps_working', eps_working, "d_low", d_low)
     59     # print(d_up, d_low)
     60     for _ in range(n_iter):



In [ ]:
starting_point_dichotomy(171, images.cpu(), labels.cpu())

tensor(0)

In [ ]:
van_model(images[171:172]).argmax()

tensor(8, device='cuda:0')

In [ ]:
labels[171:172]

tensor([8], device='cuda:0')

In [ ]:
single_compute_optimistic_radius_PGD(171, images.to(device), labels.to(device), certificates, van_model, n_iter=10)

tensor([0.0054], device='cuda:0', grad_fn=<SliceBackward0>)


tensor(0.5122, device='cuda:0')

In [ ]:
certificates[0]

tensor(0.0020, device='cuda:0', grad_fn=<SelectBackward0>)

In [ ]:
single_compute_optimistic_radius_AA(171, images.to(device), labels.to(device), certificates, model, n_iter = 10)

tensor([0.3319], device='cuda:0', grad_fn=<SqrtBackward0>)